In [4]:
# importing the necessary libraries
import pandas as pd  # for data manipulation
from matplotlib import pyplot as plt  # library for creating and visualizing plots
import numpy as np  # for numerical calculations and operations
import seaborn as sns  # for data visualization
from scipy import stats as st  # for advanced statistical calculations if needed


In [5]:
# importing the dataset
df_games = pd.read_csv('games.csv')

In [6]:
#printing the general info about the dataset
df_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  object 
 1   Platform         16715 non-null  object 
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  object 
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  object 
 10  Rating           9949 non-null   object 
dtypes: float64(6), object(5)
memory usage: 1.4+ MB


##### From the preliminary analysis of the dataset and the inspection of its structure, I have the following initial observations: 
1. There are missing values in several columns, and the score-related columns have a significant number of null entries. Since these "scores" play an important role in the required analysis, it will be necessary to define the best strategy to handle these missing values during the data‑cleaning process. 
2. Also detected integrity issues in the dataset, with many values misplaced in the wrong columns.
3. The user_score column should be converted to a float type.
4. Column names should be standardized to lowercase, and it may be necessary to rename some of them to ensure they better reflect the information they represent.

In [22]:
df_games.rename(columns={'Name': 'game_name'}, inplace=True)
#changing column names to lower case
df_games.columns = df_games.columns.str.lower()
df_games.head()

,game_name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [23]:
df_games['year_of_release'].unique() # printing the unique release year values to check how Python interprets missing and non‑numeric entries

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

##### VI noticed that Python interprets missing values and strings as NaN. In total, there are 109 misaligned rows, and 402 rows if we also count the empty cells in 'year_of_release', which is about 2.4% of a dataset with 16,715 entries. Removing these rows will not have a significant impact on the analysis.  
##### Now I will check for duplicate rows before removing the NaN values.


In [24]:
df_games.duplicated().sum() # checking for duplicate entries in the dataset

np.int64(0)

In [27]:
df_games.dropna(subset=['year_of_release'], inplace=True) # removing rows with missing values in the 'year_of_release' column
df_games['year_of_release'] = df_games['year_of_release'].astype(int) # converting the 'year_of_release' column to integer type
df_games.dropna(subset=['game_name'], inplace=True) # removing rows with missing values in the 'game_name' column
df_games.info() # checking the dataset info again to confirm the changes made to the 'game_name' column

<class 'pandas.core.frame.DataFrame'>
Index: 16444 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   game_name        16444 non-null  object 
 1   platform         16444 non-null  object 
 2   year_of_release  16444 non-null  int64  
 3   genre            16444 non-null  object 
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     7983 non-null   float64
 9   user_score       9839 non-null   object 
 10  rating           9768 non-null   object 
dtypes: float64(5), int64(1), object(5)
memory usage: 1.5+ MB


##### After the first stage of cleaning, the last three columns of the dataset (critic_score, user_score and rating) still contain missing values, which will be handled as follows:  
1. user_score – values marked as tbd will be treated as missing values (NaN), as they represent an undetermined score (absence of information). Then the data will be converted from object to float
2. The missing values in these three columns will not be removed, because dropping these rows would drastically reduce the size of the dataset and introduce significant bias, distorting subsequent analyses.  
3. The missing values will also not be filled with estimates or artificial values, as a limited analysis is preferable to a misleading one that alters means, medians and distributions, further compromising data integrity.

In [ ]:
# replacing the string 'tbd' with Nan values in the 'user_score' column to handle non‑numeric entries
df_games['user_score'] =pd.to_numeric(df_games['user_score'], errors='coerce') # converting the 'user_score' column to float type
df_games.info() # checking the dataset info again to confirm the changes made to the 'user_score' column


<class 'pandas.core.frame.DataFrame'>
Index: 16444 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   game_name        16444 non-null  object 
 1   platform         16444 non-null  object 
 2   year_of_release  16444 non-null  int64  
 3   genre            16444 non-null  object 
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     7983 non-null   float64
 9   user_score       7463 non-null   float64
 10  rating           9768 non-null   object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB
